## 5-Task MNIST Class-IL example

In [ ]:
import torch
import torch.optim as optim
import numpy as np

from networks.BP_network import BP_network
from networks.EWC_network import EWC_network
from networks.oEWC_network import oEWC_network
from networks.SI_network import SI_network
from networks.LwF_network import LwF_network, LwF_network_online
from networks.EFC_network import EFC_network
from src.dataloaders import ClassILMNIST2Task, ClassILMNIST5Task, TaskILMNIST
from src.utils import dotdict

from tqdm import tqdm

# ============================================================================
# Configuration
# ============================================================================
config = dotdict({
    "setting": "classIL5task",
    "num_tasks": 5,
    "classes_per_task": 2,
    "batch_size": 256,
    "epochs": 20,
    "loss_fn": "ce",
    "scheduler": "",
    "output_dir": "./outputs",
    "seed": 0,
    "optimizer": "Adam",
    "num_workers": 0,
    "mode": "di",
    "lr": 1e-3,
    "target_lr": 1e-1,
    "alpha_di": 0.0017,
    "alpha_I": 0.0017,
    "tau": 0.032,
    "dt_di": 0.02,
    "psi_lr": 0.1,
    "alpha_psi": 0.0,
    "time_constant_ratio": 0.2, # this param can be merged with dt_di
    "tmax_di": 40,
    "flatten_imgs": True,
    "k_p": 2.0,
    "eps": 1e-4, # there is an interplay between dt_di and eps and between target_lr and eps
    "save": False,
    "importance_ewc": 4.0,
    "beta_efc": 1e-1,
    "gamma_oewc": 1.0,
    "si_damping": 0.1,
    "normalize_fisher": True,
    "lwf_lambda": 4.0,
    "lwf_temperature": 2.0,
    "layers": [784, 400, 400, 10],
    "device": "cpu" if torch.cuda.is_available() else "cpu", # Yassine note: changed to cpu for testing
})

torch.set_default_device(config.device)
torch.manual_seed(config.seed)
np.random.seed(config.seed)

epochs_per_task = 20

# Get dataloaders for all tasks
dataloader = ClassILMNIST5Task(config)
train_loaders = []
test_loaders = []
for task_id in range(config.num_tasks):
    train_loader, test_loader = dataloader.get_dataloaders(task_id=task_id)
    train_loaders.append(train_loader)
    test_loaders.append(test_loader)
    print(f"Task {task_id}: {len(train_loader.dataset)} train, {len(test_loader.dataset)} test")

full_test_loader = test_loaders[-1]

# ============================================================================
# Helper Functions
# ============================================================================
def evaluate(network, test_loader, task_id, task_classes):
    """Evaluate network on test set for specified classes."""
    network.eval()
    correct = 0
    total = 0
    class_start, class_end = task_classes
    
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(network.device), y.to(network.device)
            labels = y.argmax(dim=1)
            
            mask = (labels >= class_start) & (labels <= class_end)
            if mask.sum() == 0:
                continue
            
            x_masked = x[mask]
            labels_masked = labels[mask]
            
            y_hat = network(x_masked)
            preds = y_hat[:, class_start:class_end+1].argmax(dim=1) + class_start
            
            correct += (preds == labels_masked).sum().item()
            total += mask.sum().item()
    
    accuracy = correct / total if total > 0 else 0.0
    return accuracy, total

def least_square_initialization(network, dataloader, task_id, classes_per_task=2, weight_decay=1e-4):
    """Least-square optimal initialization for new classifier weights."""
    network.eval()
    new_start = task_id * classes_per_task
    new_end = (task_id + 1) * classes_per_task
    
    features_list = []
    labels_list = []
    
    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(network.device)
            features = x
            for layer in network.layers[:-1]:
                features = layer(features)
            features_list.append(features)
            labels_list.append(y.argmax(dim=1))
    
    features = torch.cat(features_list, dim=0)
    labels = torch.cat(labels_list, dim=0)
    
    N, d = features.shape
    features_ext = torch.cat([features, torch.ones(N, 1, device=features.device)], dim=1)
    
    num_new_classes = new_end - new_start
    targets = torch.zeros(N, num_new_classes, device=features.device)
    for i, label in enumerate(labels):
        if new_start <= label < new_end:
            targets[i, label - new_start] = 1.0
    
    mask = (labels >= new_start) & (labels < new_end)
    features_new = features_ext[mask]
    targets_new = targets[mask]
    
    ZtZ = features_new.T @ features_new
    ZtY = features_new.T @ targets_new
    
    reg = weight_decay * features_new.shape[0] * torch.eye(d + 1, device=features.device)
    W_ls = torch.linalg.solve(ZtZ + reg, ZtY) / 2
    
    with torch.no_grad():
        for c_idx, c in enumerate(range(new_start, new_end)):
            network.layers[-1]._weights[c] = W_ls[:d, c_idx]
            network.layers[-1]._bias[c] = W_ls[d, c_idx]
    
    print(f"  LS init for heads {new_start}:{new_end}")



def evaluate_taskil(network, test_loaders, num_tasks, classes_per_task=2):
    """
    Evaluate network on Task-IL setting.
    
    In Task-IL, we evaluate each task separately using only that task's
    output head (2 classes). The task identity is known at test time.
    
    Args:
        network: The neural network
        test_loaders: List of test dataloaders, one per task
        num_tasks: Number of tasks
        classes_per_task: Classes per task (default 2)
    
    Returns:
        dict with per-task accuracies and average accuracy
    """
    network.eval()
    results = {}
    total_correct = 0
    total_samples = 0
    
    with torch.no_grad():
        for task_id in range(num_tasks):
            # Set network to evaluate on this task
            network.task_id = task_id
            
            correct = 0
            task_total = 0
            
            for x, y in test_loaders[task_id]:
                x, y = x.to(network.device), y.to(network.device)
                
                # Forward pass - network masks to current task's outputs
                y_hat = network(x)  # Shape: (batch, 2) due to task mask
                
                # y is binary one-hot (batch, 2), get class index
                labels = y.argmax(dim=1)  # 0 or 1
                preds = y_hat.argmax(dim=1)  # 0 or 1
                
                correct += (preds == labels).sum().item()
                task_total += x.size(0)
            
            accuracy = correct / task_total if task_total > 0 else 0.0
            results[f'task_{task_id}'] = accuracy
            total_correct += correct
            total_samples += task_total
    
    # Average accuracy across tasks (standard Task-IL metric)
    results['average'] = np.mean([results[f'task_{t}'] for t in range(num_tasks)])
    results['overall'] = total_correct / total_samples if total_samples > 0 else 0.0
    
    return results


def least_square_initialization_taskil(network, dataloader, task_id, classes_per_task=2, weight_decay=1e-4):
    """
    Least-square optimal initialization for new classifier weights in Task-IL.
    
    For Task-IL, we initialize the specific output head for the new task
    (2 neurons per task).
    """
    network.eval()
    new_start = task_id * classes_per_task
    new_end = (task_id + 1) * classes_per_task
    
    features_list = []
    labels_list = []
    
    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(network.device)
            # Get features from penultimate layer
            features = x
            for layer in network.layers[:-1]:
                features = layer(features)
            features_list.append(features)
            labels_list.append(y.argmax(dim=1))  # Binary: 0 or 1
    
    features = torch.cat(features_list, dim=0)
    labels = torch.cat(labels_list, dim=0)
    
    N, d = features.shape
    features_ext = torch.cat([features, torch.ones(N, 1, device=features.device)], dim=1)
    
    # Create binary targets (0 or 1 for this task)
    num_new_classes = classes_per_task
    targets = torch.zeros(N, num_new_classes, device=features.device)
    for i, label in enumerate(labels):
        targets[i, label] = 1.0
    
    # Solve least squares
    ZtZ = features_ext.T @ features_ext
    ZtY = features_ext.T @ targets
    
    reg = weight_decay * N * torch.eye(d + 1, device=features.device)
    W_ls = torch.linalg.solve(ZtZ + reg, ZtY) / 2
    
    # Initialize the task-specific output neurons
    with torch.no_grad():
        for c_idx in range(num_new_classes):
            global_idx = new_start + c_idx
            network.layers[-1]._weights[global_idx] = W_ls[:d, c_idx]
            network.layers[-1]._bias[global_idx] = W_ls[d, c_idx]
    
    print(f"  LS init for Task-IL heads {new_start}:{new_end}")


def train_continual_taskil(network_class, config, name, train_loaders, test_loaders):
    """
    Train a network on Task-IL setting.
    
    Task-IL characteristics:
    - Each task has binary classification (2 classes)
    - Task identity is known at train and test time
    - Network output is masked to current task's 2 neurons
    - Metric: average per-task accuracy
    
    Args:
        network_class: Network class to instantiate
        config: Configuration dict/object
        name: Name for logging
        train_loaders: List of training dataloaders from TaskILDataloader
        test_loaders: List of test dataloaders from TaskILDataloader
    
    Returns:
        dict with training results
    """
    print(f"\n{'='*70}")
    print(f"Training (Task-IL): {name}")
    print(f"{'='*70}")
    
    net = network_class(config).to(config.device)
    
    results = {
        'name': name,
        'setting': 'taskIL',
        'task_accuracies': [],
        'training_history': [],
    }
    
    for task_id in range(config.num_tasks):
        task_classes = [task_id * config.classes_per_task + c for c in range(config.classes_per_task)]
        print(f"\n--- Task {task_id} (classes {task_classes[0]}-{task_classes[-1]}) ---")
        
        # Set network task context
        net.task_id = task_id
        
        # Initialize new head (except first task)
        if task_id > 0:
            least_square_initialization_taskil(
                net, train_loaders[task_id], task_id, config.classes_per_task
            )
            if hasattr(net, '_first_task'):
                net._first_task = False
        
        # Setup optimizer
        optimizer = optim.Adam(net.parameters(), lr=config.lr)
        print(f"  Training for {config.epochs} epochs")
        
        # Training loop
        for epoch in range(config.epochs):
            net.train()
            pbar = tqdm(total=len(train_loaders[task_id]), 
                        desc=f"  Epoch {epoch+1}", unit="batch", leave=False)
            
            for x, y in train_loaders[task_id]:
                x, y = x.to(config.device), y.to(config.device)
                
                optimizer.zero_grad()
                y_hat = net(x)  # Masked to 2 outputs
                
                # y is binary one-hot (batch, 2)
                _ = net.calculate_loss(y_hat, y.argmax(dim=1))
                net.backward(y)
                optimizer.step()
                pbar.update(1)
            
            pbar.close()
            
            # Evaluate on all tasks seen so far
            net.eval()
            eval_results = evaluate_taskil(
                net, 
                test_loaders[:task_id + 1],  # Only evaluate on seen tasks
                task_id + 1, 
                config.classes_per_task
            )
            
            # Store history
            results['training_history'].append({
                'task_id': task_id,
                'epoch': epoch + 1,
                **eval_results
            })
            
            # Print progress
            # acc_str = " | ".join([f"T{t}={eval_results[f'task_{t}']:.3f}" 
            #                        for t in range(task_id + 1)])
            # print(f"  Epoch {epoch+1:2d}: Avg={eval_results['average']:.4f} | {acc_str}")
        
        # Complete task (update Fisher, etc.)
        if hasattr(net, 'complete_task'):
            net.complete_task(train_loaders[task_id])
        
        # Final evaluation after this task
        final_eval = evaluate_taskil(
            net, 
            test_loaders[:task_id + 1], 
            task_id + 1, 
            config.classes_per_task
        )
        final_eval['after_task'] = task_id
        results['task_accuracies'].append(final_eval)
        
        print(f"  >> Task {task_id} complete. Avg acc: {final_eval['average']:.4f}")
    
    return results

def run_taskil_experiment(network_class, config, name):
    """
    Convenience function to run a complete Task-IL experiment.
    
    Sets up dataloaders and runs training.
    """
    from src.dataloaders import TaskILMNIST
    
    # Ensure config is set for Task-IL
    original_setting = config.get('setting', 'taskIL')
    config.setting = 'taskIL'
    
    # Get dataloaders
    dataloader = TaskILMNIST(config)
    train_loaders = []
    test_loaders = []
    
    for task_id in range(config.num_tasks):
        train_loader, test_loader = dataloader.get_dataloaders(task_id=task_id)
        train_loaders.append(train_loader)
        test_loaders.append(test_loader)
        print(f"Task {task_id}: {len(train_loader.dataset)} train, {len(test_loader.dataset)} test")
    
    # Run training
    results = train_continual_taskil(network_class, config, name, train_loaders, test_loaders)
    
    # Restore original setting
    config.setting = original_setting
    
    return results


def train_continual(network_class, config, name):
    """Train a network on all 5 tasks sequentially with fixed epochs per task."""
    print(f"\n{'='*70}")
    print(f"Training: {name}")
    print(f"{'='*70}")
    
    net = network_class(config).to(config.device)
    
    results = {
        'name': name,
        'task_accuracies': [],
        'training_history': [],
    }
    
    for task_id in range(config.num_tasks):
        print(f"\n--- Task {task_id} (classes {task_id*2}-{task_id*2+1}) ---")
        
        net.task_id = task_id
        seen_classes_end = (task_id + 1) * config.classes_per_task - 1
        
        if task_id > 0:
            least_square_initialization(net, train_loaders[task_id], task_id, config.classes_per_task)
            if hasattr(net, '_first_task'):
                net._first_task = False
        
        optimizer = optim.Adam(net.parameters(), lr=config.lr)
        print(f"  Training for {config.epochs} epochs")
        
        for epoch in range(config.epochs):
            net.train()
            pbar = tqdm(total=len(train_loaders[task_id]), 
                        desc=f"  Epoch {epoch+1}", unit="batch", leave=False)
            
            for x, y in train_loaders[task_id]:
                x, y = x.to(config.device), y.to(config.device)
                optimizer.zero_grad()
                y_hat = net(x)
                _ = net.calculate_loss(y_hat, y.argmax(dim=1))
                net.backward(y)
                optimizer.step()
                pbar.update(1)
            pbar.close()
            
            net.eval()
            combined_acc, _ = evaluate(net, full_test_loader, task_id=task_id, 
                                       task_classes=[0, seen_classes_end])
            
            task_accs = {}
            for t in range(task_id + 1):
                t_start = t * config.classes_per_task
                t_end = t_start + config.classes_per_task - 1
                acc, _ = evaluate(net, full_test_loader, task_id=t, task_classes=[t_start, t_end])
                task_accs[f'task_{t}'] = acc
            
            results['training_history'].append({
                'task_id': task_id,
                'epoch': epoch + 1,
                'combined_acc': combined_acc,
                **task_accs
            })
            
            acc_str = " | ".join([f"T{t}={task_accs[f'task_{t}']:.3f}" for t in range(task_id + 1)])
            print(f"  Epoch {epoch+1:2d}: Combined={combined_acc:.4f} | {acc_str}")
        
        net.complete_task(train_loaders[task_id])
        
        final_accs = {'after_task': task_id, 'combined': combined_acc}
        for t in range(task_id + 1):
            t_start = t * config.classes_per_task
            t_end = t_start + config.classes_per_task - 1
            acc, _ = evaluate(net, full_test_loader, task_id=t, task_classes=[t_start, t_end])
            final_accs[f'task_{t}'] = acc
        results['task_accuracies'].append(final_accs)
        
        print(f"  >> Task {task_id} complete. Combined acc: {combined_acc:.4f}")
    
    return results

Task 0: 12665 train, 2115 test
Task 1: 12089 train, 4157 test
Task 2: 11263 train, 6031 test
Task 3: 12183 train, 8017 test
Task 4: 11800 train, 10000 test


In [6]:
# ============================================================================
# Run Experiments
# ============================================================================
all_results = {}

# all_results['BP'] = train_continual(BP_network, config, "BP")
# all_results['EWC'] = train_continual(EWC_network, config, "EWC")
# all_results['EFC'] = train_continual(EFC_network, config, "EFC")
all_results['EFC_inhib'] = train_continual(LwF_network_online, config, "oEWC")

# ============================================================================
# Summary
# ============================================================================
print("\n" + "="*70)
print("FINAL RESULTS SUMMARY")
print("="*70)

print("\nFinal combined accuracy (all 10 classes) after Task 4:")
print("-" * 50)
for name, res in all_results.items():
    final = res['task_accuracies'][-1]
    print(f"{res['name']:20s}: {final['combined']:.4f}")

print("\nPer-task accuracy breakdown after all tasks:")
print("-" * 70)
header = f"{'Method':<20s} | " + " | ".join([f"T{t}" for t in range(5)]) + " | Combined"
print(header)
print("-" * 70)

for name, res in all_results.items():
    final = res['task_accuracies'][-1]
    task_accs = " | ".join([f"{final[f'task_{t}']:.3f}" for t in range(5)])
    print(f"{res['name']:<20s} | {task_accs} | {final['combined']:.4f}")


Training: oEWC

--- Task 0 (classes 0-1) ---
  Training for 20 epochs


  Epoch  1: Combined=0.9962 | T0=0.996


  Epoch  2: Combined=0.9976 | T0=0.998


  Epoch  3: Combined=0.9986 | T0=0.999


  Epoch  4: Combined=0.9991 | T0=0.999


...

  Epoch 16: Combined=0.1961 | T0=0.998 | T1=0.991 | T2=0.998 | T3=0.997 | T4=0.988


  Epoch 17: Combined=0.1960 | T0=0.998 | T1=0.990 | T2=0.998 | T3=0.997 | T4=0.988


  Epoch 18: Combined=0.1961 | T0=0.997 | T1=0.990 | T2=0.998 | T3=0.997 | T4=0.989


  Epoch 19: Combined=0.1964 | T0=0.998 | T1=0.991 | T2=0.998 | T3=0.997 | T4=0.990


  Epoch 20: Combined=0.1964 | T0=0.998 | T1=0.991 | T2=0.998 | T3=0.997 | T4=0.990
  >> Task 4 complete. Combined acc: 0.1964

FINAL RESULTS SUMMARY

Final combined accuracy (all 10 classes) after Task 4:
--------------------------------------------------
oEWC                : 0.1964

Per-task accuracy breakdown after all tasks:
----------------------------------------------------------------------
Method               | T0 | T1 | T2 | T3 | T4 | Combined
----------------------------------------------------------------------
oEWC                 | 0.998 | 0.991 | 0.998 | 0.997 | 0.990 | 0.1964


In [11]:
# Method 1: Full setup
all_results = {}

all_results["BP"] = run_taskil_experiment(BP_network, config, "BP")
# all_results["EWC"] = run_taskil_experiment(EWC_network, config, "EWC")
# all_results["oEWC"] = run_taskil_experiment(oEWC_network, config, "oEWC")
# all_results["EFC"] = run_taskil_experiment(LwF_network_online, config, "EFC")

"""Print summary of Task-IL experiment results."""
print("\n" + "="*70)
print("TASK-IL RESULTS SUMMARY")
print("="*70)

print("\nFinal average accuracy across all tasks:")
print("-" * 50)
for name, res in all_results.items():
    final = res['task_accuracies'][-1]
    print(f"{res['name']:20s}: {final['average']:.4f}")

print("\nPer-task accuracy breakdown after all tasks:")
print("-" * 70)
header = f"{'Method':<20s} | " + " | ".join([f"T{t}" for t in range(5)]) + " | Avg"
print(header)
print("-" * 70)

for name, res in all_results.items():
    final = res['task_accuracies'][-1]
    task_accs = " | ".join([f"{final[f'task_{t}']:.3f}" for t in range(5)])
    print(f"{res['name']:<20s} | {task_accs} | {final['average']:.4f}")

Task 0: 12665 train, 2115 test
Task 1: 12089 train, 2042 test
Task 2: 11263 train, 1874 test
Task 3: 12183 train, 1986 test
Task 4: 11800 train, 1983 test

Training (Task-IL): BP

--- Task 0 (classes 0-1) ---


AttributeError: 'BP_network' object has no attribute 'start_task'